# Topic Identification and Classification of Short Texts
### IT5072 – Natural Language Processing · Coursework 02 · Student ID: MS26900636

**Goal.** Build a model that reads any **three-sentence text** and assigns it to one of **five classes**:
**Medical**, **Sports**, **Religion**, **Politics**, or **Other** (a text that belongs to none of the four given domains).

**Approach in one line.** Unsupervised topic modeling (LDA and NMF) is used to *discover and explain* the latent themes of the
corpora, and a supervised TF-IDF classifier with a calibrated confidence threshold is used to *decide* the final class,
including the open-set *Other* class.

| Section | Content | Marking criterion |
|---|---|---|
| 1 | Theoretical approach | Theory (30%) |
| 2 – 4 | Setup, data exploration, preprocessing | Implementation (30%) |
| 5 | Evaluation design (splits and 3-sentence chunks) | Implementation |
| 6 | Topic modeling: LDA and NMF | Implementation |
| 7 | Supervised classifiers | Implementation |
| 8 | The *Other* class: confidence threshold | Implementation / Testing |
| 9 | Final evaluation on held-out and unseen-topic data | Testing (30%) |
| 10 | Testing with hand-written outside texts | Testing (30%) |
| 11 | Discussion and conclusion | – |

**Data layout** (all paths are relative to this notebook):
```
data/
├── train/                       # given corpora, one folder per domain
│   ├── medical/sci.med/
│   ├── sports/rec.sport.baseball/, rec.sport.hockey/
│   ├── religion/soc.religion.christian/, talk.religion.misc/
│   └── politics/talk.politics.guns/, talk.politics.mideast/, talk.politics.misc/
└── outside/                     # data NOT taken from the given corpora
    ├── other_20ng/tune/         # unrelated newsgroups used to tune the Other threshold
    ├── other_20ng/test/         # different unrelated newsgroups, used only for testing
    └── handmade/<class>/        # 30 newly written 3-sentence texts (6 per class)
```

## 1. Theoretical Approach

### 1.1 Problem definition
Topic identification is the task of finding the theme of a text. Topic *modeling* answers the question
"what themes exist in this collection?" without labels, while topic *classification* answers
"which known theme does this new text belong to?". This coursework requires both:

* The corpora are already grouped by domain, so the four domains are **known classes** (a supervised, closed-set problem).
* A fifth class, **Other**, has **no training data** at all. Any text from an unseen subject (cooking, finance, technology …)
  must be recognised as "none of the above". This is an **open-set recognition** problem: the model must know when it does not know.
* Inputs are only **three sentences** long (roughly 30–70 words). Short texts are sparse: each contains few of the
  vocabulary words, so the model must rely on a small number of strong, topic-bearing keywords.

The proposed solution is a **hybrid** pipeline:

```
 raw posts ──► cleaning ──► tokens ──┬──► Bag-of-Words ──► LDA ─────────┐  (unsupervised: discover & explain topics)
                                     │                                   ├──► topic → domain mapping, topic-based classifier
                                     ├──► TF-IDF ────────► NMF ─────────┘
                                     │
                                     └──► TF-IDF (1–2-grams) ──► NB / LogReg / SVM ──► class probabilities
                                                                                              │
                                                       max probability < τ ?  ──yes──► "Other"
                                                                              └─no───► Medical / Sports / Religion / Politics
```

### 1.2 Corpus
The given corpora are eight newsgroups from the *20 Newsgroups* collection (Lang, 1995), mapped to four domains:

| Domain | Newsgroups | Posts |
|---|---|---|
| Medical | sci.med | 990 |
| Sports | rec.sport.baseball, rec.sport.hockey | 1,993 |
| Religion | soc.religion.christian, talk.religion.misc | 1,625 |
| Politics | talk.politics.guns, talk.politics.mideast, talk.politics.misc | 2,625 |

The classes are **imbalanced** (Politics has about 2.7× as many posts as Medical), which influences the choice of
evaluation metric (macro-F1) and the use of class weighting.

### 1.3 Preprocessing
Raw Usenet posts contain much text that is either noise or *leakage* (information that reveals the label but would never
appear in a real input):

| Step | Reason |
|---|---|
| Remove headers (`From`, `Organization`, `Lines`, …), keep only `Subject` | Headers name the newsgroup or the author's institution, which would let the model "cheat". The subject line is genuine topical content. |
| Remove quoted replies (`>` lines) and "*X writes:*" lines | Quoted text belongs to another post and duplicates content. |
| Remove signatures (after `--`), e-mail addresses, URLs, ASCII art | Not topical; signatures repeat across posts of the same author and cause over-fitting. |
| Lower-casing, keep alphabetic tokens of length ≥ 3 | Normalises surface forms and removes numbers and punctuation. |
| Stop-word removal (NLTK list + Usenet words like *writes*, *article*) | Function words carry no topic information. |
| Lemmatisation (WordNet) | Groups inflected forms (*doctors → doctor*, *games → game*) and reduces sparsity. Lemmatisation is chosen over stemming because it keeps real words, which makes topics easier to read. |

### 1.4 Text representation (vector space model)
Each document is represented as a vector over the vocabulary $V$.

* **Bag-of-Words (BoW)**: the count $tf(t,d)$ of each term $t$ in document $d$. Word order is ignored. LDA is defined over counts,
  so it uses BoW.
* **TF-IDF** (Salton & Buckley, 1988) re-weights counts so that words frequent in a document but rare in the collection get high weight:

$$\text{tfidf}(t,d) = \big(1 + \log tf(t,d)\big)\cdot \log\frac{1+N}{1+df(t)} + 1$$

  where $N$ is the number of documents and $df(t)$ the number of documents containing $t$. The *sublinear* $1+\log tf$ term stops
  long posts from dominating. Vectors are L2-normalised so that short (3-sentence) and long texts are comparable.
* **n-grams**: unigrams and bigrams are used for classification so that phrases such as *gun control*, *blood pressure* or *world series*
  become features.

### 1.5 Unsupervised topic modeling

**Latent Dirichlet Allocation (LDA)** (Blei, Ng & Jordan, 2003) is a generative probabilistic model. It assumes that each document
is a *mixture of topics* and each topic is a *mixture of words*:

1. For each topic $k = 1..K$: draw a word distribution $\beta_k \sim \text{Dirichlet}(\eta)$.
2. For each document $d$: draw topic proportions $\theta_d \sim \text{Dirichlet}(\alpha)$.
3. For each word position $n$ in $d$: draw a topic $z_{dn} \sim \text{Multinomial}(\theta_d)$, then the word $w_{dn} \sim \text{Multinomial}(\beta_{z_{dn}})$.

Only the words $w$ are observed. Learning reverses this process, estimating the hidden $\theta$ and $\beta$ from the observed words by
maximising $p(w \mid \alpha,\eta)$. The exact posterior is intractable, so gensim uses **online variational Bayes** (Hoffman et al., 2010).
Small Dirichlet parameters encode the belief that a document uses only a few topics and a topic uses only a few words. Here $\alpha$ and $\eta$
are learned from the data (`'auto'`).

**Choosing the number of topics $K$.** $K$ is a hyper-parameter. It is selected with **topic coherence $C_v$** (Röder et al., 2015). $C_v$
measures how often the top words of a topic co-occur in the corpus (using sliding windows, NPMI and cosine similarity). A higher value
means more human-interpretable topics.

**Non-negative Matrix Factorisation (NMF)** (Lee & Seung, 1999) is a linear-algebra alternative. The TF-IDF matrix
$V \in \mathbb{R}^{n\times m}_{\ge 0}$ is approximated by two smaller non-negative matrices:

$$V \approx W H, \qquad \min_{W,H \ge 0} \lVert V - WH\rVert_F^2$$

$W$ (documents × topics) gives each document's topic weights, and $H$ (topics × words) gives each topic's word weights. The non-negativity
makes the factorisation *additive* (parts-based), so the topics are interpretable. NMF is deterministic given its initialisation and often
gives sharper topics on short texts, while LDA gives a full probabilistic model.

**From topics to domains.** Topic models produce *unlabelled* topics. Each topic is mapped to the domain in which it has the highest
average share, $\text{domain}(k) = \arg\max_c \frac{1}{|D_c|}\sum_{d\in D_c}\theta_{dk}$. A new text is then classified by summing
its topic proportions per domain. This gives a fully **topic-based classifier** that can be compared with the supervised one.

### 1.6 Supervised classification
Because labelled data exists, supervised learners can use it directly. Three linear classifiers on TF-IDF features are compared:

* **Multinomial Naive Bayes**: applies Bayes' rule with the "naive" assumption that words are conditionally independent given the class:
  $\hat{c} = \arg\max_c \; \log P(c) + \sum_{t} x_t \log P(t \mid c)$, where $P(t \mid c)$ uses Laplace/Lidstone smoothing. It is fast and a strong baseline for text.
  The class prior $P(c)$ is set to **uniform**. With the empirical prior, the largest class (Politics) would win every uncertain
  text, including off-topic ones that should be rejected as *Other*.
* **Logistic Regression (multinomial / softmax)**: a discriminative model,
  $P(c \mid x) = \dfrac{\exp(w_c^\top x + b_c)}{\sum_j \exp(w_j^\top x + b_j)}$, trained by minimising regularised cross-entropy.
  It outputs well-calibrated probabilities, which matter for the *Other* decision.
* **Linear Support Vector Machine**: finds the maximum-margin hyperplane (hinge loss + L2 penalty). It is often the most accurate model
  on high-dimensional sparse text. SVMs give scores rather than probabilities, so **Platt scaling** (sigmoid calibration, Platt, 1999) is applied with
  `CalibratedClassifierCV`.

`class_weight='balanced'` is used where supported to counter the class imbalance.

A fourth, **hybrid** model, *LDA topic proportions → Logistic Regression*, tests whether the compact topic representation
($K$ features instead of ~100k) is enough for classification.

### 1.7 Detecting the *Other* class (open-set recognition)
No training examples exist for *Other*, and "everything else" cannot be sampled completely. The model therefore uses a **reject option**
based on its confidence (the maximum-softmax-probability baseline of Hendrycks & Gimpel, 2017):

$$\hat{y}(x) = \begin{cases} \arg\max_c P(c\mid x) & \text{if } \max_c P(c\mid x) \ge \tau \\ \text{Other} & \text{otherwise}\end{cases}$$

A text about cooking contains few Medical/Sports/Religion/Politics keywords, so the probability mass is spread out and the maximum is low.
The threshold $\tau$ is **tuned** on a validation set that contains in-domain texts and *Other* texts from three unrelated newsgroups
(computer graphics, X-Windows, cars). It is then **tested** on four *different* unrelated newsgroups (space, for-sale, motorcycles,
electronics) and on hand-written texts. This shows the rule generalises to subjects it never saw, instead of memorising one kind of "other".

### 1.8 Evaluation protocol
* The given posts are split **at document level** into train (70%), validation (15%) and test (15%), stratified by domain.
  Splitting before chunking guarantees that no sentence from a test post was seen in training.
* Models are **trained on whole posts** (more context gives better word statistics) but **evaluated on three-sentence chunks** cut
  from the body of validation/test posts. This matches the real task. The subject line is excluded from chunks because real inputs have none.
* The validation set is used for every choice (model selection, $\tau$). The test set is used **once**, for the final numbers.
* Metrics: accuracy, per-class precision/recall/F1, **macro-F1** (the unweighted mean of class F1s, so the small Medical and *Other*
  classes count as much as Politics), and confusion matrices.

### References
* Blei, D. M., Ng, A. Y., & Jordan, M. I. (2003). Latent Dirichlet Allocation. *Journal of Machine Learning Research*, 3, 993–1022.
* Hendrycks, D., & Gimpel, K. (2017). A Baseline for Detecting Misclassified and Out-of-Distribution Examples in Neural Networks. *ICLR*.
* Hoffman, M., Bach, F., & Blei, D. (2010). Online Learning for Latent Dirichlet Allocation. *NeurIPS*.
* Lang, K. (1995). NewsWeeder: Learning to Filter Netnews. *ICML*.
* Lee, D. D., & Seung, H. S. (1999). Learning the parts of objects by non-negative matrix factorization. *Nature*, 401, 788–791.
* Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.
* Platt, J. (1999). Probabilistic outputs for support vector machines. *Advances in Large Margin Classifiers*.
* Röder, M., Both, A., & Hinneburg, A. (2015). Exploring the Space of Topic Coherence Measures. *WSDM*.
* Salton, G., & Buckley, C. (1988). Term-weighting approaches in automatic text retrieval. *Information Processing & Management*, 24(5).
* Sievert, C., & Shirley, K. (2014). LDAvis: A method for visualizing and interpreting topics. *ACL Workshop on Interactive Language Learning*.

## 2. Setup
Libraries, random seeds (for reproducible results) and NLTK resources. Exact package versions are listed in `requirements.txt`.

In [ ]:
import re, random, warnings, time
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import NMF
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

for pkg in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

DATA = Path('data')
DOMAINS = ['medical', 'sports', 'religion', 'politics']
CLASSES = DOMAINS + ['other']
PALETTE = dict(zip(CLASSES, sns.color_palette('Set2', 5)))

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 110)
print('Setup complete.')

## 3. Data Loading and Exploration
Every file is read from `data/train/<domain>/<newsgroup>/`. The folder name gives the label.

In [ ]:
def read_file(path):
    return Path(path).read_text(encoding='latin-1')

rows = []
for domain in DOMAINS:
    for group_dir in sorted((DATA / 'train' / domain).iterdir()):
        for f in sorted(group_dir.iterdir()):
            rows.append({'domain': domain, 'newsgroup': group_dir.name,
                         'file': f.name, 'raw': read_file(f)})
df = pd.DataFrame(rows)

print(f'{len(df):,} posts loaded\n')
summary = (df.groupby(['domain', 'newsgroup']).size().rename('posts').reset_index())
display(summary)
display(df.groupby('domain').size().rename('posts').to_frame().T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
dom_counts = df['domain'].value_counts().reindex(DOMAINS)
axes[0].bar(dom_counts.index, dom_counts.values, color=[PALETTE[d] for d in DOMAINS])
axes[0].set_title('Posts per domain (class balance)')
axes[0].set_ylabel('posts')
for i, v in enumerate(dom_counts.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center')

ng = df.groupby(['newsgroup', 'domain']).size().reset_index(name='posts').sort_values('posts')
axes[1].barh(ng['newsgroup'], ng['posts'], color=[PALETTE[d] for d in ng['domain']])
axes[1].set_title('Posts per newsgroup (colour = domain)')
plt.tight_layout(); plt.show()

A raw post, showing why cleaning is required: header fields, an e-mail address, a quoted reply and a signature block surround the actual content.

In [ ]:
print(df.loc[df['newsgroup'] == 'sci.med', 'raw'].iloc[3][:1500])

## 4. Preprocessing
Implementation of the cleaning steps described in §1.3:
1. `split_post`: separate the header from the body and keep only the `Subject`.
2. `clean_body`: drop quoted lines, attribution lines, signatures, ASCII art, e-mails and URLs.
3. `preprocess`: lowercase → tokenise → remove stop-words → lemmatise.

In [ ]:
HEADER_SPLIT = re.compile(r'\n\s*\n')
QUOTE_LINE   = re.compile(r'^\s*(>|\|>|\||:)')
ATTRIB_LINE  = re.compile(r'(writes|wrote|says)\s*:\s*$|^\s*in article\s*<', re.I)
EMAIL        = re.compile(r'\S+@\S+')
URL          = re.compile(r'(https?://|www\.)\S+')

def split_post(raw):
    # Return (subject, body). Header = everything before the first blank line.
    parts = HEADER_SPLIT.split(raw, maxsplit=1)
    header, body = (parts[0], parts[1]) if len(parts) == 2 else ('', raw)
    m = re.search(r'^Subject:\s*(.*)$', header, re.M | re.I)
    subject = re.sub(r'^(re:\s*)+', '', m.group(1).strip(), flags=re.I) if m else ''
    return subject, body

def clean_body(body):
    kept = []
    for line in body.splitlines():
        s = line.strip()
        if s in ('--', '-- '):                       # signature delimiter: stop reading
            break
        if QUOTE_LINE.match(line) or ATTRIB_LINE.search(line):
            continue                                 # quoted reply / "X writes:"
        if s and sum(c.isalpha() for c in s) < 0.5 * len(s):
            continue                                 # separators, ASCII art, tables of numbers
        kept.append(s)
    text = ' '.join(kept)
    text = EMAIL.sub(' ', text)
    text = URL.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

lemmatizer = WordNetLemmatizer()
STOP = set(stopwords.words('english')) | {
    'writes', 'wrote', 'article', 'would', 'could', 'also', 'one', 'like', 'get', 'know',
    'think', 'say', 'said', 'make', 'well', 'much', 'even', 'way', 'thing', 'really',
    'see', 'go', 'going', 'want', 'may', 'might', 'must', 'many', 'use', 'used', 'anyone',
    'something', 'someone', 'etc', 'please', 'thanks', 'thank', 'yes', 'lot', 'got',
    'two', 'first', 'take', 'come', 'still', 'look', 'good', 'right', 'time', 'year',
    'mean', 'point', 'another', 'without', 'though', 'since', 'let', 'need', 'actually'}
TOKEN = re.compile(r'[a-z]+')

@lru_cache(maxsize=None)
def lemma(word):
    return lemmatizer.lemmatize(word)

def preprocess(text):
    # Text -> list of clean lemmatised tokens.
    out = []
    for tok in TOKEN.findall(text.lower()):
        if len(tok) < 3 or tok in STOP:
            continue
        lem = lemma(tok)
        if lem not in STOP and len(lem) >= 3:
            out.append(lem)
    return out

In [ ]:
t0 = time.time()
df[['subject', 'body_raw']] = df['raw'].apply(lambda r: pd.Series(split_post(r)))
df['body'] = df['body_raw'].map(clean_body)
df['tokens'] = (df['subject'] + '. ' + df['body']).map(preprocess)
df['n_tokens'] = df['tokens'].map(len)
before = len(df)
df = df[df['n_tokens'] >= 5].reset_index(drop=True)      # drop empty posts (e.g. only quoted text)
print(f'Preprocessed in {time.time()-t0:.1f}s. Kept {len(df):,} of {before:,} posts '
      f'({before-len(df)} posts had < 5 content words after cleaning).')

ex = df[df['newsgroup'] == 'sci.med'].iloc[3]
print('\nSUBJECT :', ex['subject'])
print('BODY    :', ex['body'][:500])
print('TOKENS  :', ex['tokens'][:40])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=df, x='domain', y='n_tokens', order=DOMAINS, palette=PALETTE, ax=axes[0], showfliers=False)
axes[0].set_title('Content words per post after cleaning (outliers hidden)')
vocab_by_domain = {d: len(set(w for toks in df.loc[df.domain == d, 'tokens'] for w in toks)) for d in DOMAINS}
axes[1].bar(vocab_by_domain.keys(), vocab_by_domain.values(), color=[PALETTE[d] for d in DOMAINS])
axes[1].set_title('Vocabulary size per domain')
plt.tight_layout(); plt.show()
print(df.groupby('domain')['n_tokens'].describe()[['mean', '50%', 'max']].round(1))

**Most frequent words per domain.** Even before any modeling, the vocabularies are clearly different. This suggests that
keyword-based (bag-of-words) models should separate the domains well.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
for ax, d in zip(axes, DOMAINS):
    top = pd.Series([w for toks in df.loc[df.domain == d, 'tokens'] for w in toks]).value_counts().head(15)
    ax.barh(top.index[::-1], top.values[::-1], color=PALETTE[d])
    ax.set_title(d.capitalize())
plt.suptitle('Top 15 words per domain', y=1.02); plt.tight_layout(); plt.show()

## 5. Evaluation Design: Splits and Three-Sentence Chunks
* **Document-level split** 70 / 15 / 15 (train / validation / test), stratified by domain.
* **Chunks:** each validation/test post body is split into sentences (NLTK Punkt) and grouped into consecutive windows of **three
  sentences**, which mirrors the real input. Chunks with fewer than 5 content words are dropped, and at most 5 chunks are taken
  per post so that very long posts do not dominate.
* **Other data:** posts from unrelated newsgroups are cleaned the same way and chunked identically.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['domain'], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df['domain'], random_state=SEED)
print(f'train {len(train_df):,} | validation {len(val_df):,} | test {len(test_df):,} posts')

def make_chunks(body, rng, n=3, max_chunks=5, min_tokens=5):
    sents = [s for s in sent_tokenize(body) if len(s.split()) >= 3]
    chunks = [' '.join(sents[i:i + n]) for i in range(0, len(sents), n)]
    chunks = [c for c in chunks if len(preprocess(c)) >= min_tokens]
    if len(chunks) > max_chunks:
        chunks = rng.sample(chunks, max_chunks)
    return chunks

def chunk_frame(frame):
    rng = random.Random(SEED)
    out = [{'label': r.domain, 'source': r.newsgroup, 'text': c}
           for r in frame.itertuples() for c in make_chunks(r.body, rng)]
    return pd.DataFrame(out)

def load_other(split):
    rows = []
    for g in sorted((DATA / 'outside' / 'other_20ng' / split).iterdir()):
        for f in sorted(g.iterdir()):
            _, body = split_post(read_file(f))
            rows.append({'domain': 'other', 'newsgroup': g.name, 'body': clean_body(body)})
    return pd.DataFrame(rows)

val_chunks, test_chunks = chunk_frame(val_df), chunk_frame(test_df)
other_tune_chunks, other_test_chunks = chunk_frame(load_other('tune')), chunk_frame(load_other('test'))

sizes = pd.DataFrame({
    'validation (in-domain)': val_chunks['label'].value_counts(),
    'test (in-domain)': test_chunks['label'].value_counts()}).reindex(DOMAINS)
sizes.loc['other'] = [len(other_tune_chunks), len(other_test_chunks)]
display(sizes.rename(columns={'validation (in-domain)': 'validation chunks (Other = tune groups)',
                              'test (in-domain)': 'test chunks (Other = unseen groups)'}))
print('Other tune groups:', sorted(other_tune_chunks.source.unique()))
print('Other test groups:', sorted(other_test_chunks.source.unique()))
print('\nExample 3-sentence test chunk:\n ', test_chunks.sample(1, random_state=1)['text'].iloc[0])

## 6. Unsupervised Topic Modeling

### 6.1 LDA: choosing the number of topics with coherence
The Bag-of-Words dictionary keeps words that appear in at least 5 training posts and in at most 50% of them, since very rare and
very common words carry little topic information. LDA is trained for several values of $K$, and the $C_v$ coherence of each model is measured.

In [ ]:
dictionary = Dictionary(train_df['tokens'])
dictionary.filter_extremes(no_below=5, no_above=0.5, keep_n=15000)
bow_train = [dictionary.doc2bow(t) for t in train_df['tokens']]
print(f'Dictionary: {len(dictionary):,} terms, {len(bow_train):,} training documents')

def train_lda(k):
    return LdaModel(bow_train, id2word=dictionary, num_topics=k, passes=10, iterations=100,
                    alpha='auto', eta='auto', chunksize=2000, random_state=SEED)

K_GRID = [4, 6, 8, 10, 12, 16, 20]
lda_models, coherence = {}, {}
t0 = time.time()
for k in K_GRID:
    lda_models[k] = train_lda(k)
    coherence[k] = CoherenceModel(model=lda_models[k], texts=train_df['tokens'].tolist(),
                                  dictionary=dictionary, coherence='c_v', processes=1).get_coherence()
    print(f'K={k:>2}  coherence C_v = {coherence[k]:.4f}')
print(f'({time.time()-t0:.0f}s)')

In [ ]:
BEST_K = max(coherence, key=coherence.get)
lda = lda_models[BEST_K]

plt.figure(figsize=(7, 4))
plt.plot(list(coherence), list(coherence.values()), 'o-', color='#4c72b0')
plt.axvline(BEST_K, ls='--', color='grey')
plt.title(f'LDA topic coherence vs number of topics (best K = {BEST_K})')
plt.xlabel('number of topics K'); plt.ylabel('$C_v$ coherence'); plt.show()

### 6.2 What the LDA topics contain, and how they map to domains
Each topic is mapped to the domain where it has the highest average share (§1.5). The heat-map shows that mapping:
each row is a topic, and a dark cell means the topic is used mostly by that domain's documents.

In [ ]:
def lda_theta(model, bows):
    # Document-topic matrix theta (n_docs x K).
    theta = np.zeros((len(bows), model.num_topics))
    for i, b in enumerate(bows):
        for k, p in model.get_document_topics(b, minimum_probability=0):
            theta[i, k] = p
    return theta

def topic_domain_matrix(theta, labels):
    labels = np.asarray(labels)
    M = np.vstack([theta[labels == d].mean(axis=0) for d in DOMAINS]).T     # K x 4
    return pd.DataFrame(M, columns=DOMAINS)

theta_train = lda_theta(lda, bow_train)
lda_map_df = topic_domain_matrix(theta_train, train_df['domain'])
lda_topic2domain = lda_map_df.idxmax(axis=1)

lda_topics = pd.DataFrame({
    'top words': [', '.join(w for w, _ in lda.show_topic(k, topn=10)) for k in range(BEST_K)],
    'mapped domain': lda_topic2domain.values})
lda_topics.index.name = 'topic'
display(lda_topics)

plt.figure(figsize=(6, 0.45 * BEST_K + 1.5))
sns.heatmap(lda_map_df.div(lda_map_df.sum(axis=1), axis=0), annot=True, fmt='.2f', cmap='Blues', cbar=False)
plt.title('LDA: share of each topic by domain (row-normalised)'); plt.ylabel('topic'); plt.show()

In [ ]:
cols = 4
rows_ = int(np.ceil(BEST_K / cols))
fig, axes = plt.subplots(rows_, cols, figsize=(4.5 * cols, 3.2 * rows_))
for k, ax in enumerate(axes.flat):
    ax.axis('off')
    if k >= BEST_K:
        continue
    freqs = dict(lda.show_topic(k, topn=40))
    colour = PALETTE[lda_topic2domain[k]]
    hexc = '#%02x%02x%02x' % tuple(int(255 * c) for c in colour)
    wc = WordCloud(width=450, height=300, background_color='white', random_state=SEED,
                   color_func=lambda *a, **kw: hexc).generate_from_frequencies(freqs)
    ax.imshow(wc); ax.set_title(f'Topic {k} → {lda_topic2domain[k]}')
plt.suptitle('LDA topic word clouds (colour = mapped domain)', y=1.01); plt.tight_layout(); plt.show()

### 6.3 Interactive topic map (pyLDAvis)
Each circle is a topic. Its area is the topic's prevalence, and the distance between circles reflects how different the topics' word
distributions are (Sievert & Shirley, 2014). Select a topic to see its most relevant words. Topics from the same domain sit close together.

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()
lda_vis = gensimvis.prepare(lda, bow_train, dictionary, sort_topics=False, mds='mmds')
lda_vis

### 6.4 NMF topics
NMF is fitted on a TF-IDF matrix with the same number of topics, so the two topic models can be compared directly.

In [ ]:
train_df['text_pp'] = train_df['tokens'].str.join(' ')
nmf_vec = TfidfVectorizer(token_pattern=r'\S+', min_df=5, max_df=0.5, max_features=15000)
X_nmf = nmf_vec.fit_transform(train_df['text_pp'])
nmf = NMF(n_components=BEST_K, init='nndsvda', max_iter=500, random_state=SEED)
W_train = nmf.fit_transform(X_nmf)
nmf_words = np.array(nmf_vec.get_feature_names_out())

def normalise_rows(W):
    s = W.sum(axis=1, keepdims=True)
    return np.divide(W, s, out=np.full_like(W, 1 / W.shape[1]), where=s > 0)

nmf_map_df = topic_domain_matrix(normalise_rows(W_train), train_df['domain'])
nmf_topic2domain = nmf_map_df.idxmax(axis=1)
nmf_topics = pd.DataFrame({
    'top words': [', '.join(nmf_words[np.argsort(h)[::-1][:10]]) for h in nmf.components_],
    'mapped domain': nmf_topic2domain.values})
nmf_topics.index.name = 'topic'
display(nmf_topics)

### 6.5 Topic models as classifiers
A three-sentence chunk is turned into topic proportions. The proportions of all topics mapped to the same domain are summed, and the
domain with the largest total wins. This is evaluated on the **validation chunks** (closed-set, 4 domains).

In [ ]:
def topic_scores(theta, topic2domain):
    # Sum topic proportions per mapped domain -> (n x 4) score matrix.
    S = np.zeros((theta.shape[0], len(DOMAINS)))
    for k, d in topic2domain.items():
        S[:, DOMAINS.index(d)] += theta[:, k]
    return S

def lda_scores(texts):
    bows = [dictionary.doc2bow(preprocess(t)) for t in texts]
    return topic_scores(lda_theta(lda, bows), lda_topic2domain)

def nmf_scores(texts):
    W = nmf.transform(nmf_vec.transform([' '.join(preprocess(t)) for t in texts]))
    return topic_scores(normalise_rows(W), nmf_topic2domain)

topic_results = {}
for name, fn in [('LDA (topic → domain)', lda_scores), ('NMF (topic → domain)', nmf_scores)]:
    pred = np.array(DOMAINS)[fn(val_chunks['text']).argmax(axis=1)]
    topic_results[name] = {'accuracy': accuracy_score(val_chunks['label'], pred),
                           'macro-F1': f1_score(val_chunks['label'], pred, average='macro')}
display(pd.DataFrame(topic_results).T.round(3))